### Text-to-SQL
In this notebook we illustrate a mechanism to convert plain text to SQL using the LangChain framework. We have developed this on a sufficiently complext database schema - the `sakila` schema, which is a DVD rental application. This is compatible with `MySQL` or `PostgreSQL` or `SQLite`. We'll use `SQLite` to keep things simple.

Install the following packages using your environment manager (e.g. `uv`)
```bash
$> uv add python_dotenv rich sqlparse tabulate langchain_core langchain_community
```

In [154]:
import os
from pathlib import Path
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from sqlalchemy import text
from tabulate import tabulate

from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, Runnable
from langchain_core.output_parsers import StrOutputParser
from langchain_community.utilities import SQLDatabase
from langchain.prompts import ChatPromptTemplate

In [155]:
# to suppress specific warnings I get from SQLAlchemy
import warnings
from sqlalchemy.exc import SAWarning

# Suppress the specific SAWarning related to unresolvable cycles
warnings.filterwarnings(
    "ignore",
    category=SAWarning,
    message="Cannot correctly sort tables; there are unresolvable cycles between tables .*",
)

In [156]:
# load all API keys from .env file
load_dotenv(override=True)
# for colorful text
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [157]:
# create our LLM - we'll be using Gemini-2.5-flash, but you can use any
# llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
llm = init_chat_model("claude-3-7-sonnet-20250219", model_provider="anthropic")

In [158]:
# load the SQLite database
db_path = Path(os.getcwd()) / ".." / "db" / "sakila_master.db"
db = None

if not db_path.exists():
    raise FileNotFoundError(f"Database file not found at {db_path}")
else:
    print(f"Loading database from {db_path}")
    db = SQLDatabase.from_uri(f"sqlite:///{db_path}")

Loading database from c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\Advanced RAG\..\db\sakila_master.db


In [159]:
def get_schema():
    """Get the database schema as a string."""
    schema = db.get_table_info()
    return schema


def run_query(sql_query: str):
    """Run a SQL query against the database and return the results."""
    results = db.run(sql_query)
    return results


class SQLQueryExecutor(Runnable):
    """Runnable to execute the SQL query against the database."""

    def __init__(self, db: SQLDatabase):
        self.db = db

    def invoke(self, sql_query: str, config=None):
        """
        Executes the SQL query string against the database.
        Returns a tuple: (column_names, results)
        """
        try:
            # Use SQLDatabase.run to execute the query
            # NOTE: db.run returns the results as a string.
            # We must use the underlying connection for structured results.

            # Get the underlying connection
            with self.db._engine.connect() as connection:
                # Execute the query
                # result = connection.execute(sqlparse.parse(sql_query)[0])
                result = connection.execute(text(sql_query))

                # Fetch results
                column_names = list(result.keys())
                rows = result.fetchall()

                return (column_names, rows)
        except Exception as e:
            # Important to catch errors during execution
            # return (["Error"], [f"SQL Execution Failed: {e}"])
            return f"SQL_EXECUTION_ERROR: {e}"

In [160]:
from langchain.schema import BaseOutputParser
import sqlparse
import re


class SQLParserAndFormatter(BaseOutputParser[str]):
    """Parse generated SQL and neatly format it"""

    def parse(self, text: str) -> str:
        # Use regex to replace all sequences of whitespace (including newlines)
        # with a single space, and then strip leading/trailing spaces.
        pre_formatted_sql = re.sub(r"\s+", " ", text).strip()
        # format it using sqlparse
        formatted_sql = sqlparse.format(
            pre_formatted_sql,
            reindent=True,
            keyword_case="upper",
            indent_width=4,
        )
        return formatted_sql.strip()


class MarkdownTableParser(BaseOutputParser[str]):
    """Output parser that converts (columns, rows) tuple OR error string into a Markdown table."""

    def parse(self, data) -> str:
        # 1. Check if the input is the string error flag from the Executor
        if isinstance(data, str) and data.startswith("SQL_EXECUTION_ERROR:"):
            error_message = data.replace("SQL_EXECUTION_ERROR: ", "")
            return f"**Error Executing Query:**\n```\n{error_message}\n```"

        # 2. Proceed with successful tuple parsing (original logic)
        if not isinstance(data, tuple) or len(data) != 2:
            return "Error: Invalid data format passed to MarkdownTableParser."

        column_names, rows = data

        if not rows:
            return "**No results found for the query.**"

        # ... (tabulate code unchanged) ...
        table = tabulate(
            rows,
            headers=column_names,
            tablefmt="pipe",
        )
        return table

In [161]:
template = """Based on the table schema below, write a SQL query that would 
nswer the user's question:

{schema}

Question: {question}
SQL Query:"""

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given an input question, convert it to a SQL query."
            "Please do not return anything else apart from the SQL query, no prefix or suffix quotes, no additional text apart from SQL query. Keep all SQL keywords in uppercase",
        ),
        ("human", template),
    ]
)


def text_to_sql(llm, prompt=prompt) -> Runnable:
    """converts user-query in natural languaue (text) to SQL based on database schema"""

    return (
        RunnablePassthrough.assign(schema=lambda x: get_schema())
        | prompt
        | llm
        | StrOutputParser()
        | SQLParserAndFormatter()
    )


def execute_and_format_query(sql_query: str) -> str:
    """
    Combines the SQL execution and the Markdown table formatting.
    This bypasses the tuple-output issue by keeping the logic inside a single step.
    """

    # Instantiate the executor and run it
    executor = SQLQueryExecutor(db)
    results_tuple_or_error_string = executor.invoke(sql_query)

    # Instantiate the formatter and parse the executor's result
    formatter = MarkdownTableParser()
    final_output = formatter.parse(results_tuple_or_error_string)

    return final_output


def text_to_sql_and_run(llm, prompt=prompt) -> Runnable:
    """converts user-query in natural languaue (text) to SQL based on database schema"""

    text_to_sql_chain = (
        RunnablePassthrough.assign(schema=lambda x: get_schema())
        | prompt
        | llm
        | StrOutputParser()
        | SQLParserAndFormatter()
        # The output here is the formatted SQL string
    )

    # --- CORRECTED CHAIN ---
    # Replace the two separate custom steps with the single RunnableLambda
    execution_step = RunnableLambda(execute_and_format_query)
    execution_chain = text_to_sql_chain | execution_step

    return execution_chain

In [162]:
# user_query = "List the names of all actors (last name & first name) that acted in film AMADEUS HOLY"
# generated_sql = text_to_sql(llm, prompt).invoke({"question": user_query})
# print(generated_sql)
# print(type(generated_sql))

Some queries you can try (in natural language):
1. List the names of all actors (last name & first name) that acted in film AMADEUS HOLY
2. In which stores is the film AMADEUS HOLY available? List the store ID, address and manager name
3. List names and date rented of all customers who have rented the movie AMADEUS HOLY along with the store they rented from (store id and address)
4. Show me by store ID and address how many times the movie AMADEUS HOLY has been rented out?

In [168]:
user_query = "Show me by store ID and address how many times the movie AMADEUS HOLY has been rented out?"
generated_sql = text_to_sql(llm, prompt).invoke({"question": user_query})
response = text_to_sql_and_run(llm, prompt).invoke({"question": user_query})
console.print(f"\n[bold green]SQL:[/bold green]\n{generated_sql}")
console.print(f"\n[bold blue]Result:[/bold blue]")
console.print(Markdown(response))


SQL:
SELECT store.store_id,
       address.address,
       COUNT(*) AS rental_count
FROM store
JOIN address ON store.address_id = address.address_id
JOIN inventory ON store.store_id = inventory.store_id
JOIN film ON inventory.film_id = film.film_id
JOIN rental ON inventory.inventory_id = rental.inventory_id
WHERE film.title = 'AMADEUS HOLY'
GROUP BY store.store_id,
         address.address

Result:

                                                
  store_id   address              rental_count  
 ────────────────────────────────────────────── 
         1   47 MySakila Drive              13  
         2   28 MySQL Boulevard              8  
                                                
